In [1]:
%load_ext autoreload
%autoreload 2
import sys, json
sys.path.insert(0, "..")
from pathlib import Path
import numpy as np, pandas as pd, joblib
from sklearn.metrics import accuracy_score, f1_score, recall_score, classification_report, confusion_matrix
from src.experiments import log_experiment

DATA, MODELS, REPORTS = Path("../data/processed"), Path("../models"), Path("../reports")
gold = pd.read_csv("../data/gold/gold_labeled.csv")
gold = gold[gold["intent"].fillna("").ne("") & gold["priority"].fillna("").ne("")]
weak = pd.read_parquet(DATA / "tickets_weak_labels.parquet")
g = (gold[["tweet_id", "intent", "priority"]]
     .rename(columns={"intent": "gold_intent", "priority": "gold_priority"})
     .merge(weak, on="tweet_id", how="left"))
assert g["text_clean"].notna().all()

print(f"Gold rows: {len(g)}")
print("Intent:  ", g["gold_intent"].value_counts().to_dict())
print("Priority:", g["gold_priority"].value_counts().to_dict())

Gold rows: 400
Intent:   {'General': 124, 'Complaint': 123, 'Technical': 98, 'Billing': 35, 'Account': 20}
Priority: {'High': 158, 'Medium': 158, 'Low': 84}


In [2]:
from gensim.models import Word2Vec
import gensim.downloader as api
from tensorflow import keras
from tensorflow.keras import layers

def cls_scores(y, p):
    return {"accuracy": accuracy_score(y, p), "macro_f1": f1_score(y, p, average="macro", zero_division=0)}

# 1) How good were the labeling methods?
fired = g["rule_label"].notna()
agree = fired & (g["rule_label"] == g["zs_label"])
print("LABEL SOURCES vs gold                 n     acc    macro-F1")
for name, mask, col in [("Rules (where a rule fired)", fired, "rule_label"),
                        ("Zero-shot DeBERTa (all rows)", g.index == g.index, "zs_label"),
                        ("Silver: rules = zero-shot", agree, "rule_label")]:
    m = cls_scores(g.loc[mask, "gold_intent"], g.loc[mask, col])
    print(f"{name:32} {mask.sum():4}   {m['accuracy']:.3f}   {m['macro_f1']:.3f}")

# 2) PS-1 models
X = g["text_clean"]
arr = lambda s: np.array(s.tolist(), dtype=object)

def mean_embed(texts, kv, dim=100):
    out = np.zeros((len(texts), dim), dtype=np.float32)
    for i, t in enumerate(texts):
        vecs = [kv[w] for w in t.split() if w in kv]
        if vecs:
            out[i] = np.mean(vecs, axis=0)
    return out

w2v = Word2Vec.load(str(MODELS / "w2v_tickets.model")).wv
glove = api.load("glove-twitter-100")                     # cached from earlier, loads in seconds
vocab1 = json.loads((MODELS / "ps1_vocab.json").read_text())
vec1 = layers.TextVectorization(max_tokens=None, output_sequence_length=40, standardize=None,
                                vocabulary=vocab1[2:])   # Keras re-adds "" and [UNK] at 0 and 1
assert vec1.get_vocabulary() == vocab1
le1 = joblib.load(MODELS / "ps1_label_encoder.joblib")
bilstm1 = keras.models.load_model(MODELS / "ps1_bilstm.keras")

preds = {
    "TF-IDF + LR": joblib.load(MODELS / "ps1_tfidf_lr.joblib").predict(X),
    "Word2Vec + LR": joblib.load(MODELS / "ps1_word2vec_lr.joblib").predict(mean_embed(X.tolist(), w2v)),
    "GloVe + LR": joblib.load(MODELS / "ps1_glove-twitter-100_lr.joblib").predict(mean_embed(X.tolist(), glove)),
    "BiLSTM (W2V-init)": le1.inverse_transform(bilstm1.predict(vec1(arr(X)).numpy(), verbose=0).argmax(axis=1)),
}
silver_f1 = {"TF-IDF + LR": 0.971, "Word2Vec + LR": 0.824, "GloVe + LR": 0.783, "BiLSTM (W2V-init)": 0.976}

print("\nPS-1 MODELS              gold acc  gold macro-F1  (silver test F1)")
for name, p in preds.items():
    m = cls_scores(g["gold_intent"], p)
    log_experiment("PS-1", name, {"eval": "gold"}, m)
    print(f"{name:22}   {m['accuracy']:.3f}     {m['macro_f1']:.3f}        ({silver_f1[name]:.3f})")

best1 = max(preds, key=lambda k: cls_scores(g["gold_intent"], preds[k])["macro_f1"])
print(f"\nBest on gold: {best1}\n")
print(classification_report(g["gold_intent"], preds[best1], digits=3, zero_division=0))
labels = sorted(g["gold_intent"].unique())
print(pd.DataFrame(confusion_matrix(g["gold_intent"], preds[best1], labels=labels),
                   index=[f"true_{l}" for l in labels], columns=labels))

# Error analysis file for the report
g["ps1_pred"] = preds[best1]
err = g.loc[g["ps1_pred"] != g["gold_intent"], ["tweet_id", "text", "gold_intent", "ps1_pred"]]
err.to_csv(REPORTS / "ps1_gold_errors.csv", index=False)
print(f"\n{len(err)} errors saved → reports/ps1_gold_errors.csv")

LABEL SOURCES vs gold                 n     acc    macro-F1
Rules (where a rule fired)        223   0.605   0.581
Zero-shot DeBERTa (all rows)      400   0.522   0.470
Silver: rules = zero-shot         126   0.714   0.670

PS-1 MODELS              gold acc  gold macro-F1  (silver test F1)
TF-IDF + LR              0.585     0.538        (0.971)
Word2Vec + LR            0.515     0.474        (0.824)
GloVe + LR               0.527     0.484        (0.783)
BiLSTM (W2V-init)        0.557     0.532        (0.976)

Best on gold: TF-IDF + LR

              precision    recall  f1-score   support

     Account      0.385     0.250     0.303        20
     Billing      0.590     0.657     0.622        35
   Complaint      0.635     0.439     0.519       123
     General      0.532     0.798     0.639       124
   Technical      0.688     0.541     0.606        98

    accuracy                          0.585       400
   macro avg      0.566     0.537     0.538       400
weighted avg      0.600 

In [3]:
NUM_COLS = ["vader", "n_exclaim", "caps_ratio", "escalation_kw", "risk_kw", "calm_kw"]
gp = g.copy()
gp[NUM_COLS] = gp[NUM_COLS].astype(float)
gp["n_exclaim"] = gp["n_exclaim"].clip(upper=5)

def p_scores(y, p):
    return {"accuracy": accuracy_score(y, p),
            "macro_f1": f1_score(y, p, average="macro", zero_division=0),
            "high_recall": recall_score(y, p, labels=["High"], average="macro", zero_division=0)}

# 1) Label sources
agree_p = gp["rule_priority"] == gp["zs_priority"]
print("LABEL SOURCES vs gold          n     acc   macro-F1  High recall")
for name, mask, col in [("Rules (all rows)", gp.index == gp.index, "rule_priority"),
                        ("Zero-shot (all rows)", gp.index == gp.index, "zs_priority"),
                        ("Silver: rules = zero-shot", agree_p, "rule_priority")]:
    m = p_scores(gp.loc[mask, "gold_priority"], gp.loc[mask, col])
    print(f"{name:26} {mask.sum():4}   {m['accuracy']:.3f}   {m['macro_f1']:.3f}     {m['high_recall']:.3f}")

# 2) PS-2 models
aux = joblib.load(MODELS / "ps2_bilstm_aux.joblib")
vocab2 = json.loads((MODELS / "ps2_vocab.json").read_text())
vec2 = layers.TextVectorization(max_tokens=None, output_sequence_length=40, standardize=None,
                                vocabulary=vocab2[2:])
assert vec2.get_vocabulary() == vocab2
bilstm2 = keras.models.load_model(MODELS / "ps2_bilstm.keras")
X2 = {"text": vec2(arr(gp["text_clean"])).numpy(),
      "signals": aux["scaler"].transform(gp[NUM_COLS]).astype("float32")}

preds2 = {
    "LR text+signals": joblib.load(MODELS / "ps2_best_classical.joblib").predict(gp),
    "BiLSTM+signals": aux["label_encoder"].inverse_transform(bilstm2.predict(X2, verbose=0).argmax(axis=1)),
}
silver2 = {"LR text+signals": (0.894, 0.878), "BiLSTM+signals": (0.883, 0.899)}

print("\nPS-2 MODELS          gold macro-F1  gold High recall  (silver F1 / High recall)")
for name, p in preds2.items():
    m = p_scores(gp["gold_priority"], p)
    log_experiment("PS-2", name, {"eval": "gold"}, m)
    print(f"{name:18}      {m['macro_f1']:.3f}          {m['high_recall']:.3f}          "
          f"({silver2[name][0]:.3f} / {silver2[name][1]:.3f})")

print("\nPredicted vs gold class mix:")
mix = pd.DataFrame({"gold": gp["gold_priority"].value_counts(normalize=True),
                    **{n: pd.Series(p).value_counts(normalize=True) for n, p in preds2.items()}}).round(3)
print(mix)

best2 = max(preds2, key=lambda k: p_scores(gp["gold_priority"], preds2[k])["macro_f1"])
print(f"\nBest on gold: {best2}\n")
print(classification_report(gp["gold_priority"], preds2[best2], digits=3, zero_division=0))
order = ["High", "Medium", "Low"]
print(pd.DataFrame(confusion_matrix(gp["gold_priority"], preds2[best2], labels=order),
                   index=[f"true_{l}" for l in order], columns=order))

g["ps2_pred"] = preds2[best2]
g.loc[g["ps2_pred"] != g["gold_priority"], ["tweet_id", "text", "gold_priority", "ps2_pred"]].to_csv(
    REPORTS / "ps2_gold_errors.csv", index=False)

LABEL SOURCES vs gold          n     acc   macro-F1  High recall
Rules (all rows)            400   0.378   0.344     0.082
Zero-shot (all rows)        400   0.378   0.374     0.234
Silver: rules = zero-shot   233   0.395   0.339     0.039

PS-2 MODELS          gold macro-F1  gold High recall  (silver F1 / High recall)
LR text+signals         0.345          0.082          (0.894 / 0.878)
BiLSTM+signals          0.368          0.108          (0.883 / 0.899)

Predicted vs gold class mix:
         gold  LR text+signals  BiLSTM+signals
High    0.395            0.062           0.072
Low     0.210            0.418           0.408
Medium  0.395            0.520           0.520

Best on gold: BiLSTM+signals

              precision    recall  f1-score   support

        High      0.586     0.108     0.182       158
         Low      0.356     0.690     0.470        84
      Medium      0.399     0.525     0.454       158

    accuracy                          0.395       400
   macro avg      0

In [4]:
from sklearn.model_selection import train_test_split

dev_idx, test_idx = train_test_split(gp.index, test_size=0.6, stratify=gp["gold_priority"], random_state=42)
pd.DataFrame({"tweet_id": gp["tweet_id"], "gold_split": np.where(gp.index.isin(dev_idx), "dev", "test")}).to_csv(
    "../data/gold/gold_split.csv", index=False)
print(f"Gold dev {len(dev_idx)} | gold test {len(test_idx)}")

lr2 = joblib.load(MODELS / "ps2_best_classical.joblib")
proba = {
    "LR text+signals": pd.DataFrame(lr2.predict_proba(gp), columns=lr2.classes_, index=gp.index),
    "BiLSTM+signals": pd.DataFrame(bilstm2.predict(X2, verbose=0),
                                   columns=aux["label_encoder"].classes_, index=gp.index),
}

def decide(P, t):
    """High if P(High) >= t, else the likelier of Medium / Low."""
    return np.where(P["High"] >= t, "High", P[["Medium", "Low"]].idxmax(axis=1))

y_dev, y_test = gp.loc[dev_idx, "gold_priority"], gp.loc[test_idx, "gold_priority"]
ths = np.round(np.arange(0.02, 0.61, 0.02), 2)
print("\n                     threshold | gold-TEST macro-F1 | High recall | High precision")
for name, P in proba.items():
    t_best = max(ths, key=lambda t: p_scores(y_dev, decide(P.loc[dev_idx], t))["macro_f1"])
    p_def, p_cal = P.loc[test_idx].idxmax(axis=1), decide(P.loc[test_idx], t_best)
    for label, p in [("default (argmax)", p_def), (f"tuned t={t_best:.2f}", p_cal)]:
        m = p_scores(y_test, p)
        prec = (p[p == "High"] == y_test[p == "High"]).mean() if (p == "High").any() else 0
        print(f"{name:16} {label:18} {m['macro_f1']:.3f}              {m['high_recall']:.3f}        {prec:.3f}")
    log_experiment("PS-2", name, {"eval": "gold_test", "high_threshold": float(t_best)},
                   p_scores(y_test, decide(P.loc[test_idx], t_best)))

# PS-1 re-reported on the same gold-test rows, so all final numbers share one test set
m1 = cls_scores(g.loc[test_idx, "gold_intent"], g.loc[test_idx, "ps1_pred"])
print(f"\nPS-1 {best1} on gold-test: acc {m1['accuracy']:.3f} | macro-F1 {m1['macro_f1']:.3f}")

Gold dev 160 | gold test 240

                     threshold | gold-TEST macro-F1 | High recall | High precision
LR text+signals  default (argmax)   0.356              0.095        0.600
LR text+signals  tuned t=0.02       0.381              0.200        0.514
BiLSTM+signals   default (argmax)   0.381              0.105        0.625
BiLSTM+signals   tuned t=0.02       0.411              0.200        0.594

PS-1 TF-IDF + LR on gold-test: acc 0.579 | macro-F1 0.544


In [7]:
dev = gp.loc[dev_idx].copy()
is_high = dev["gold_priority"].eq("High")
print(f"Base High rate on gold-dev: {is_high.mean():.2f}\n")

signals = {
    "vader <= -0.6":         dev["vader"] <= -0.6,
    "vader <= -0.3":         dev["vader"] <= -0.3,
    "vader < 0":             dev["vader"] < 0,
    "escalation_kw":         dev["escalation_kw"].astype(bool),
    "risk_kw":               dev["risk_kw"].astype(bool),
    "n_exclaim >= 1":        dev["n_exclaim"] >= 1,
    "caps_ratio >= 0.3":     dev["caps_ratio"] >= 0.3,
    "zs intent = Complaint": dev["zs_label"].eq("Complaint"),
    "zs priority = High":    dev["zs_priority"].eq("High"),
}
print("signal                  fires   P(High | fires)   share of High caught")
for name, m in signals.items():
    print(f"{name:22}  {m.mean():5.2f}        {is_high[m].mean() if m.any() else 0:.2f}              "
          f"{(m & is_high).sum() / is_high.sum():.2f}")

def rule_v2(d, neg):
    """Candidate v2: ANY single urgency signal → High (v1 needed two)."""
    risk = d["risk_kw"].astype(bool)
    esc = d["escalation_kw"].astype(bool)
    calm = d["calm_kw"].astype(bool)
    high = risk | esc | (d["vader"] <= neg) | (d["caps_ratio"] >= 0.3) | (d["n_exclaim"] >= 2)
    positive = d["vader"] >= 0.4
    calm_ok = calm & (d["vader"] >= 0)
    general_ok = (d["zs_label"] == "General") & (d["vader"] >= -0.2)
    low = ~high & (positive | calm_ok | general_ok)
    return np.select([high, low], ["High", "Low"], default="Medium")

Base High rate on gold-dev: 0.39

signal                  fires   P(High | fires)   share of High caught
vader <= -0.6            0.15        0.42              0.16
vader <= -0.3            0.31        0.54              0.43
vader < 0                0.40        0.53              0.54
escalation_kw            0.06        0.80              0.13
risk_kw                  0.01        0.50              0.02
n_exclaim >= 1           0.19        0.50              0.24
caps_ratio >= 0.3        0.01        0.50              0.02
zs intent = Complaint    0.36        0.40              0.37
zs priority = High       0.21        0.53              0.29


In [8]:
print("Rule on gold-dev                macro-F1   High recall   predicted High share")
m_v1 = p_scores(dev["gold_priority"], dev["rule_priority"])
v1_share = (dev["rule_priority"] == "High").mean()
print(f"v1 (current rules)              {m_v1['macro_f1']:.3f}      {m_v1['high_recall']:.3f}         {v1_share:.2f}")

for neg in [-0.6, -0.4, -0.3, -0.2, -0.05]:
    p = rule_v2(dev, neg)
    m = p_scores(dev["gold_priority"], p)
    share = (p == "High").mean()
    print(f"v2, vader <= {neg:<6}             {m['macro_f1']:.3f}      {m['high_recall']:.3f}         {share:.2f}")

Rule on gold-dev                macro-F1   High recall   predicted High share
v1 (current rules)              0.305      0.063         0.06
v2, vader <= -0.6               0.402      0.349         0.25
v2, vader <= -0.4               0.435      0.556         0.38
v2, vader <= -0.3               0.428      0.556         0.38
v2, vader <= -0.2               0.433      0.619         0.42
v2, vader <= -0.05              0.415      0.635         0.45


In [9]:
from sklearn.base import clone
from src.labeling import rule_priority_v2

# Relabel the 100k (gold excluded) with v2
weak = pd.read_parquet(DATA / "tickets_weak_labels.parquet")
sp2 = weak[~weak["tweet_id"].isin(set(gold["tweet_id"]))].copy()
sp2[NUM_COLS] = sp2[NUM_COLS].astype(float)
sp2["n_exclaim"] = sp2["n_exclaim"].clip(upper=5)
sp2["priority"] = rule_priority_v2(sp2, sp2["zs_label"])
print("Silver v2 class mix:", sp2["priority"].value_counts(normalize=True).round(3).to_dict())
sp2.to_parquet(DATA / "silver_priority_v2.parquet", index=False)

# Retrain the same LR text+signals pipeline on v2 labels
tr2, va2 = train_test_split(sp2, test_size=0.1, stratify=sp2["priority"], random_state=42)
lr_v2 = clone(lr2).fit(tr2, tr2["priority"])
print("Silver-val (v2 labels):", {k: round(v, 3) for k, v in p_scores(va2["priority"], lr_v2.predict(va2)).items()})

# Tune High threshold on gold-dev, then score everything ONCE on gold-test
P2 = pd.DataFrame(lr_v2.predict_proba(gp), columns=lr_v2.classes_, index=gp.index)
ths2 = np.round(np.arange(0.10, 0.81, 0.05), 2)
t2 = max(ths2, key=lambda t: p_scores(y_dev, decide(P2.loc[dev_idx], t))["macro_f1"])

candidates = {
    "Rules v1":                gp.loc[test_idx, "rule_priority"],
    "BiLSTM v1 (tuned t)":     decide(proba["BiLSTM+signals"].loc[test_idx], 0.02),
    "Rules v2 (no model)":     rule_priority_v2(gp.loc[test_idx], gp.loc[test_idx, "zs_label"]),
    "LR v2 (argmax)":          lr_v2.predict(gp.loc[test_idx]),
    f"LR v2 (tuned t={t2:.2f})": decide(P2.loc[test_idx], t2),
}
print("\nGOLD-TEST (240 rows)       macro-F1   High recall   accuracy")
for name, p in candidates.items():
    m = p_scores(y_test, p)
    print(f"{name:24}   {m['macro_f1']:.3f}      {m['high_recall']:.3f}        {m['accuracy']:.3f}")
    log_experiment("PS-2", name, {"eval": "gold_test", "labels": "v2" if "v2" in name else "v1"}, m)

joblib.dump({"model": lr_v2, "high_threshold": float(t2), "num_cols": NUM_COLS},
            MODELS / "ps2_lr_v2.joblib")

Silver v2 class mix: {'High': 0.424, 'Low': 0.334, 'Medium': 0.242}
Silver-val (v2 labels): {'accuracy': 0.855, 'macro_f1': 0.839, 'high_recall': 0.932}

GOLD-TEST (240 rows)       macro-F1   High recall   accuracy
Rules v1                   0.370      0.095        0.404
BiLSTM v1 (tuned t)        0.411      0.200        0.421
Rules v2 (no model)        0.435      0.558        0.446
LR v2 (argmax)             0.422      0.558        0.433
LR v2 (tuned t=0.65)       0.431      0.526        0.438


['../models/ps2_lr_v2.joblib']

In [10]:
joblib.dump({"model": lr_v2, "high_threshold": None, "num_cols": NUM_COLS}, MODELS / "ps2_lr_v2.joblib")

['../models/ps2_lr_v2.joblib']